# Auditoría de variables candidatas para un Índice de Vulnerabilidad Laboral Relativa (IVLR) en el empleo informal

## Notebook metodológico del Trabajo de Fin de Máster

Documenta de manera reproducible la auditoría de variables candidatas que podrán integrar, en una etapa posterior, un Índice de Vulnerabilidad Laboral Relativa (IVLR) dentro del empleo informal.

### Alcance del presente documento

- Restringe el análisis al universo de personas ocupadas con empleo informal entre 2023T4 y 2025T4.
- Audita variables candidatas y describe su disponibilidad, cobertura, universo de aplicación y comparabilidad temporal.
- No construye el IVLR, no asigna pesos, no define categorías de vulnerabilidad y no modifica los estados laborales ni los perfiles de clustering.
- `cluster_final` se emplea únicamente como variable descriptiva de contexto y no podrá formar parte del futuro índice.

La unidad de análisis es la observación persona-trimestre.


In [84]:
# 1. Configuración local y dependencias

import importlib.util
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import scipy
import sklearn
from scipy.stats import chi2_contingency
from IPython.display import display

# Verificación de dependencias (sin instalar automáticamente)
_REQUERIDOS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
}
_faltantes = [
    pkg for mod, pkg in _REQUERIDOS.items()
    if importlib.util.find_spec(mod) is None
]
if _faltantes:
    raise ImportError(
        "El notebook requiere dependencias que no están en el entorno "
        "Conda activo:\n  " + "\n".join(f"• {p}" for p in _faltantes)
        + "\n\nInstálelas manualmente: conda install "
        + " ".join(_faltantes)
        + "\n\nEl notebook se detiene aquí. No se modifica el entorno."
    )

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.getenv(
        "TFM_PROJECT_ROOT",
        r"C:\Users\tomas\OneDrive\Master Granada\TFM"
    )
).expanduser().resolve()
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT no encontrado: {PROJECT_ROOT}"

DATA_PATH = (
    PROJECT_ROOT
    / "Salidas"
    / "07_clustering"
    / "base_final_clusters_sin_region.parquet"
)
ZIP_PATH = (
    PROJECT_ROOT
    / "Datos"
    / "Bases EPH"
    / "bases_eph_TFM.zip"
)
OUT_DIR = PROJECT_ROOT / "Salidas" / "08_vulnerabilidad"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"DATA_PATH no encontrado: {DATA_PATH}"
assert ZIP_PATH.exists(), f"ZIP_PATH no encontrado: {ZIP_PATH}"

versiones_entorno = pd.DataFrame({
    "componente": [
        "Python", "intérprete", "pandas", "numpy", "scipy",
        "scikit-learn", "pyarrow", "fuente de datos utilizada",
    ],
    "version": [
        sys.version.split()[0], sys.executable, pd.__version__, np.__version__,
        scipy.__version__, sklearn.__version__, pyarrow.__version__, "",
    ],
})
display(versiones_entorno)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH:    {DATA_PATH}")
print(f"OUT_DIR:      {OUT_DIR}")



,componente,version
0,Python,3.11.14
1,intérprete,c:\Users\tomas\.conda\envs\trabajo\python.exe
2,pandas,3.0.5
3,numpy,2.3.5
4,scipy,1.16.3
5,scikit-learn,1.9.0
6,pyarrow,23.0.1
7,fuente de datos utilizada,


PROJECT_ROOT: C:\Users\tomas\OneDrive\Master Granada\TFM
DATA_PATH:    C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\07_clustering\base_final_clusters_sin_region.parquet
OUT_DIR:      C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\08_vulnerabilidad


## 2. Carga del universo informal

El universo se toma de `base_final_clusters_sin_region.parquet`, que contiene las 80.300 observaciones persona-trimestre de personas ocupadas con empleo informal entre 2023T4 y 2025T4, junto con el perfil de clustering asignado. Este notebook no reestima FAMD ni K-means: únicamente emplea la base como fuente y conserva `id_persona`, `ANO4`, `TRIMESTRE`, `PONDERA` y `cluster_final` como variables de contexto.


In [85]:
def cargar_universo(path):
    """Carga el universo de trabajo (parquet -> csv) verificando integridad."""
    try:
        return pd.read_parquet(path), "parquet"
    except Exception as e:
        print(f"Aviso pd.read_parquet: {e}")
        csv = path.with_suffix(".csv")
        assert csv.exists(), f"No hay CSV equivalente en {csv}"
        return pd.read_csv(csv), "csv_fallback"

base, fuente_universo = cargar_universo(DATA_PATH)
versiones_entorno.loc[
    versiones_entorno["componente"] == "fuente de datos utilizada",
    "version",
] = fuente_universo
print(f"Fuente del universo: {fuente_universo}")

METADATOS = ["id_persona", "ANO4", "TRIMESTRE", "PONDERA"]
CONTEXTO = ["cluster_final", "REGION", "CAES", "CAT_OCUP", "TAM_EST"]
ACTIVAS_BASE = ["CH04", "CH06", "NIVEL_ED", "INTENSI", "log_P21_real"]

print("Dimensiones:", base.shape)
print("Columnas:", list(base.columns))
print("Periodos:", sorted(base.groupby(["ANO4", "TRIMESTRE"]).size().index.tolist()))
print("Cluster:", base["cluster_final"].value_counts(dropna=False).sort_index().to_dict())

assert len(base) == 80300
assert base.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum() == 0



Fuente del universo: parquet
Dimensiones: (80300, 14)
Columnas: ['id_persona', 'ANO4', 'TRIMESTRE', 'PONDERA', 'CH04', 'CH06', 'NIVEL_ED', 'REGION', 'CAT_OCUP', 'CAES', 'TAM_EST', 'INTENSI', 'log_P21_real', 'cluster_final']
Periodos: [(2023, 4), (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2025, 1), (2025, 2), (2025, 3), (2025, 4)]
Cluster: {np.int64(1): 24571, np.int64(2): 19286, np.int64(3): 5783, np.int64(4): 9867, np.int64(5): 20244, <NA>: 549}


## 3. Recuperación de variables EPH

La base de clustering contiene las variables analíticas y de contexto ya consolidadas. Las variables candidatas adicionales (ingreso de la ocupación principal, horas trabajadas, deseo de trabajar más horas, antigüedad, continuidad, temporalidad, protección social y variables propias de los trabajadores independientes) se encuentran en las bases usuarias EPH originales contenidas en `Datos/Bases EPH/bases_eph_TFM.zip`.

Para cada trimestre de la ventana 2023T4-2025T4 se lee el archivo individual correspondiente, se construye `id_persona` con la misma regla del proyecto (`CODUSU_NRO_HOGAR_COMPONENTE`) y se incorporan las variables de interés mediante un emparejamiento 1:1 por `(id_persona, ANO4, TRIMESTRE)`.


In [86]:
import zipfile
import io

ZIP_PATH = PROJECT_ROOT / "Datos" / "Bases EPH" / "bases_eph_TFM.zip"

VENTANA = {
    (2023, 4): "usu_individual_T423.txt",
    (2024, 1): "usu_individual_T124.txt",
    (2024, 2): "usu_individual_T224.txt",
    (2024, 3): "usu_individual_T324.txt",
    (2024, 4): "usu_individual_T424.txt",
    (2025, 1): "usu_individual_T125.txt",
    (2025, 2): "usu_individual_T225.txt",
    (2025, 3): "usu_individual_T325.txt",
    (2025, 4): "usu_individual_T425.txt",
}

VARS_EPH = [
    # Ingreso
    "P21", "P47T", "IPCF",
    # Horas y subempleo
    "PP3E_TOT", "PP3F_TOT", "PP03G", "PP03H", "PP03I",
    # Pluriempleo
    "PP03C", "PP03D",
    # Antigüedad y continuidad
    "PP05H", "PP07A", "PP04B3_MES", "PP04B3_ANO",
    # Temporalidad / trabajo ocasional
    "PP07C", "PP07D", "PP07E",
    # Protección y beneficios (asalariados)
    "PP07F1", "PP07G1", "PP07H", "PP07I",
    # Independientes: registro y capacidad productiva
    "PP05I", "PP05J", "PP05E", "PP05F", "PP05C_1", "PP05C_2", "PP05C_3",
    "PP06A", "PP06E", "PP06H",
]

def leer_eph(name):
    with zipfile.ZipFile(ZIP_PATH).open(name) as f:
        raw = f.read().decode("latin-1")
    df = pd.read_csv(io.StringIO(raw), sep=";", dtype=str)
    df.columns = [c.strip('"') for c in df.columns]
    return df

def construir_id(df):
    return (
        df["CODUSU"].astype(str) + "_"
        + df["NRO_HOGAR"].astype(str) + "_"
        + df["COMPONENTE"].astype(str)
    )

frames = []
for (a, t), fname in VENTANA.items():
    d = leer_eph(fname)
    d["id_persona"] = construir_id(d)
    keep = [c for c in ["id_persona", "EMPLEO"] + VARS_EPH if c in d.columns]
    d = d[keep]
    d["ANO4"] = a
    d["TRIMESTRE"] = t
    frames.append(d)
    print(f"{fname}: {len(d)} filas")

eph = pd.concat(frames, ignore_index=True)
print("\nEPH consolidado:", eph.shape)



usu_individual_T423.txt: 47337 filas
usu_individual_T124.txt: 46050 filas
usu_individual_T224.txt: 47151 filas
usu_individual_T324.txt: 47564 filas
usu_individual_T424.txt: 46860 filas
usu_individual_T125.txt: 45425 filas
usu_individual_T225.txt: 46086 filas
usu_individual_T325.txt: 44946 filas
usu_individual_T425.txt: 43703 filas

EPH consolidado: (415122, 35)


## 4. Validación del emparejamiento

Se verifica que la base de clustering se pueda emparejar 1:1 con las variables extraídas de los archivos EPH, y que se conserven las 80.300 observaciones informales.


In [87]:
base_aud = base.merge(
    eph,
    on=["id_persona", "ANO4", "TRIMESTRE"],
    how="left",
    validate="1:1"
)
print("Base auditada:", base_aud.shape)
assert len(base_aud) == 80300
assert base_aud.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum() == 0
assert (base_aud["EMPLEO"] == "2").sum() == 80300

control_merge = pd.DataFrame({
    "indicador": [
        "filas_base",
        "filas_base_auditada",
        "claves_duplicadas",
        "empleo_informal_confirmado"
    ],
    "valor": [
        len(base),
        len(base_aud),
        base_aud.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum(),
        (base_aud["EMPLEO"] == "2").sum()
    ]
})
display(control_merge)



Base auditada: (80300, 46)


,indicador,valor
0,filas_base,80300
1,filas_base_auditada,80300
2,claves_duplicadas,0
3,empleo_informal_confirmado,80300


## 5. Diccionario de universos y códigos por variable (con filtros condicionales)

Se define el universo de aplicación (general, asalariados, independientes), los códigos de no respuesta, los códigos de salto (no aplica) y, cuando corresponde, un filtro condicional adicional (por ejemplo, para preguntas que solo aplican si se respondió afirmativamente a una pregunta anterior). Esta información es la base de la auditoría de cobertura.


In [88]:
# CORRECCIÓN 1: normalizar() con <NA> y sin HTML
def normalizar(serie):
    """Normaliza cadenas sin convertir los faltantes en texto."""
    return (
        serie.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        })
    )

# Máscaras de universo (sobre la base auditada)
CAT_OCUP_NUM = pd.to_numeric(base_aud["CAT_OCUP"], errors="coerce")
ES_OCUPADO_INFORMAL = pd.Series(True, index=base_aud.index)
ES_INDEPENDIENTE = CAT_OCUP_NUM.isin([1, 2])
ES_ASALARIADO = CAT_OCUP_NUM.eq(3)
ES_PATRON = CAT_OCUP_NUM.eq(1)
ES_CUENTAPROPISTA = CAT_OCUP_NUM.eq(2)
ES_FAMILIAR = CAT_OCUP_NUM.eq(4)

MASCARA_UNIVERSO = {
    "todos": ES_OCUPADO_INFORMAL,
    "independientes": ES_INDEPENDIENTE,
    "asalariados": ES_ASALARIADO,
    "patrones": ES_PATRON,
    "cuentapropistas": ES_CUENTAPROPISTA,
    "familiares": ES_FAMILIAR,
}

# Diccionario con codigos_validos y estado_filtro
UNIVERSOS = {
    "P21": {
        "universo": "todos",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": None,
        "codigos_no_respuesta": {"-9"},
        "codigos_no_aplicacion": set(),
        "codificacion": "confirmada",
    },
    "PP3E_TOT": {
        "universo": "todos",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": None,
        "codigos_no_respuesta": {"999"},
        "codigos_no_aplicacion": set(),
        "codificacion": "pendiente de confirmación documental",
    },
    "PP03G": {
        "universo": "todos",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP03H": {
        "universo": "todos",
        "filtro_condicional": lambda df: (
            normalizar(df["PP03G"]) == "1"
        ),
        "estado_filtro": "confirmado",
        "codigos_validos": {"1", "2", "3"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP03I": {
        "universo": "todos",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP05H": {
        "universo": "independientes",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2", "3", "4", "5", "6"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "pendiente de confirmación documental",
    },
    "PP07A": {
        "universo": "asalariados",
        "filtro_condicional": None,
        "estado_filtro": (
            "confirmado documentalmente; implementación pendiente"
        ),
        "observacion": (
            "Excluir del universo condicional a los casos derivados desde "
            "servicio doméstico a 7f, una vez identificado de forma "
            "inequívoca el código correspondiente en la base."
        ),
        "codigos_validos": {"1", "2", "3", "4", "5", "6"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP07C": {
        "universo": "asalariados",
        "filtro_condicional": None,
        "estado_filtro": (
            "confirmado documentalmente; implementación pendiente"
        ),
        "observacion": (
            "Excluir del universo condicional a los casos derivados desde "
            "servicio doméstico a 7f, una vez identificado de forma "
            "inequívoca el código correspondiente en la base."
        ),
        "codigos_validos": {"1", "2"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP07D": {
        "universo": "asalariados",
        "filtro_condicional": lambda df: (
            normalizar(df["PP07C"]).isin({"1", "9"})
        ),
        "estado_filtro": "confirmado",
        "codigos_validos": {"1", "2", "3", "4", "5"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP07H": {
        "universo": "asalariados",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "confirmada",
    },
    "PP05I": {
        "universo": "independientes",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2", "3", "4"},
        "codigos_no_respuesta": {"9"},
        "codigos_no_aplicacion": {"0"},
        "codificacion": "pendiente de confirmación documental",
    },
    "NIVEL_ED": {
        "universo": "todos",
        "filtro_condicional": None,
        "estado_filtro": "no requiere",
        "codigos_validos": {"1", "2", "3", "4", "5", "6", "7"},
        "codigos_no_respuesta": set(),
        "codigos_no_aplicacion": set(),
        "codificacion": "confirmada",
    },
}



## 6. Auditoría de universos y filtros (incluyendo códigos inesperados)

Para cada variable se calcula:
- n_total
- n_universo_general
- n_fuera_universo
- n_universo_condicional
- n_salto_interno
- n_no_respuesta
- n_no_aplica_interno
- n_faltante_real
- n_codigo_inesperado
- n_respuesta_valida
- pct_cobertura_universo

Se verifican las identidades para asegurar que la descomposición es exhaustiva.


In [92]:
def auditar_variable(col):
    info = UNIVERSOS.get(col)
    if info is None:
        return None

    s = normalizar(base_aud[col])
    mask_general = MASCARA_UNIVERSO[info["universo"]]

    # Aplicar filtro condicional si existe
    filtro = info["filtro_condicional"]
    if filtro is None:
        mask_condicional = mask_general.copy()
    else:
        mask_condicional = mask_general & filtro(base_aud)

    no_resp = info["codigos_no_respuesta"]
    no_aplica = info["codigos_no_aplicacion"]
    codigos_validos = info.get("codigos_validos")

    n_total = len(base_aud)
    n_universo_general = int(mask_general.sum())
    n_universo_condicional = int(mask_condicional.sum())
    n_fuera_universo = n_total - n_universo_general
    n_salto_interno = n_universo_general - n_universo_condicional

    # Sobre el universo condicional
    s_cond = s[mask_condicional]
    n_no_respuesta = int(s_cond.isin(no_resp).sum())
    n_no_aplica_interno = int(s_cond.isin(no_aplica).sum())
    n_faltante_real = int(s_cond.isna().sum())

    # Clasificación de códigos válidos e inesperados
    if codigos_validos is None:
        mask_valida = ~(s_cond.isna() | s_cond.isin(no_resp) | s_cond.isin(no_aplica))
        mask_inesperado = pd.Series(False, index=s_cond.index, dtype="boolean")
    else:
        mask_valida = s_cond.isin(codigos_validos)
        mask_inesperado = (
            s_cond.notna()
            & ~s_cond.isin(codigos_validos)
            & ~s_cond.isin(no_resp)
            & ~s_cond.isin(no_aplica)
        )

    n_respuesta_valida = int(mask_valida.sum())
    n_codigo_inesperado = int(mask_inesperado.sum())

    pct_cobertura = round(100 * n_respuesta_valida / n_universo_condicional, 2) if n_universo_condicional > 0 else np.nan

    # Verificaciones de integridad
    assert n_universo_general + n_fuera_universo == n_total
    assert n_universo_condicional + n_salto_interno == n_universo_general
    assert (
        n_respuesta_valida
        + n_no_respuesta
        + n_no_aplica_interno
        + n_faltante_real
        + n_codigo_inesperado
        == n_universo_condicional
    )

    return {
        "variable": col,
        "universo_aplicable": info["universo"],
        "tiene_filtro_condicional": filtro is not None,
        "estado_filtro": info["estado_filtro"],
        "n_total": n_total,
        "n_universo_general": n_universo_general,
        "n_fuera_universo": n_fuera_universo,
        "n_universo_condicional": n_universo_condicional,
        "n_salto_interno": n_salto_interno,
        "n_no_respuesta": n_no_respuesta,
        "n_no_aplica_interno": n_no_aplica_interno,
        "n_faltante_real": n_faltante_real,
        "n_codigo_inesperado": n_codigo_inesperado,
        "n_respuesta_valida": n_respuesta_valida,
        "pct_cobertura_universo": pct_cobertura,
        "codificacion": info["codificacion"],
    }

# Variables a auditar (las que tienen entrada en UNIVERSOS)
variables_auditar = list(UNIVERSOS.keys())
resultados_auditoria = []
for v in variables_auditar:
    res = auditar_variable(v)
    if res is not None:
        resultados_auditoria.append(res)

tabla_auditoria = pd.DataFrame(resultados_auditoria)
display(tabla_auditoria)

# Exportar
tabla_auditoria.to_csv(OUT_DIR / "auditoria_codigos_y_universos.csv", index=False, encoding="utf-8-sig")



,variable,universo_aplicable,tiene_filtro_condicional,estado_filtro,n_total,n_universo_general,n_fuera_universo,n_universo_condicional,n_salto_interno,n_no_respuesta,n_no_aplica_interno,n_faltante_real,n_codigo_inesperado,n_respuesta_valida,pct_cobertura_universo,codificacion
0,P21,todos,False,no requiere,80300,80300,0,80300,0,13989,0,0,0,66311,82.58,confirmada
1,PP3E_TOT,todos,False,no requiere,80300,80300,0,80300,0,176,0,0,0,80124,99.78,pendiente de confirmación documental
2,PP03G,todos,False,no requiere,80300,80300,0,80300,0,33,1,0,0,80266,99.96,confirmada
3,PP03H,todos,True,confirmado,80300,80300,0,21817,58483,113,0,0,0,21704,99.48,confirmada
4,PP03I,todos,False,no requiere,80300,80300,0,80300,0,206,1,0,0,80093,99.74,confirmada
5,PP05H,independientes,False,no requiere,80300,30017,50283,30017,0,129,0,0,0,29888,99.57,pendiente de confirmación documental
6,PP07A,asalariados,False,confirmado documentalmente; implementación pen...,80300,49514,30786,49514,0,254,10284,0,0,38976,78.72,confirmada
7,PP07C,asalariados,False,confirmado documentalmente; implementación pen...,80300,49514,30786,49514,0,8241,10284,0,0,30989,62.59,confirmada
8,PP07D,asalariados,True,confirmado,80300,49514,30786,18889,30625,13563,0,0,0,5326,28.20,confirmada
9,PP07H,asalariados,False,no requiere,80300,49514,30786,49514,0,0,0,0,0,49514,100.00,confirmada


## 7. Cobertura temporal (por trimestre)

Se calcula la cobertura de respuesta válida para cada variable dentro de su universo condicional, desagregado por trimestre. Esto permite detectar cambios en la disponibilidad de las variables a lo largo del tiempo.


In [90]:
def cobertura_por_periodo(col):
    info = UNIVERSOS.get(col)
    if info is None:
        return pd.DataFrame()

    s = normalizar(base_aud[col])
    mask_general = MASCARA_UNIVERSO[info["universo"]]
    filtro = info["filtro_condicional"]
    if filtro is None:
        mask_condicional = mask_general.copy()
    else:
        mask_condicional = mask_general & filtro(base_aud)

    no_resp = info["codigos_no_respuesta"]
    no_aplica = info["codigos_no_aplicacion"]
    codigos_validos = info.get("codigos_validos")

    filas = []
    for (a, t), g in base_aud.groupby(["ANO4", "TRIMESTRE"]):
        idx = g.index
        gen = mask_general.loc[idx]
        cond = mask_condicional.loc[idx]
        n_general = int(gen.sum())
        n_condicional = int(cond.sum())
        n_salto = n_general - n_condicional

        s_cond = s.loc[idx[cond]]
        if n_condicional == 0:
            filas.append({
                "ANO4": a,
                "TRIMESTRE": t,
                "n_universo_general": n_general,
                "n_universo_condicional": 0,
                "n_salto_interno": n_salto,
                "n_respuesta_valida": 0,
                "n_no_respuesta": 0,
                "n_no_aplica_interno": 0,
                "n_faltante_real": 0,
                "n_codigo_inesperado": 0,
                "pct_cobertura": np.nan,
                "estado_filtro": info["estado_filtro"],
            })
            continue

        n_no_resp = int(s_cond.isin(no_resp).sum())
        n_no_aplica = int(s_cond.isin(no_aplica).sum())
        n_faltante = int(s_cond.isna().sum())

        if codigos_validos is None:
            mask_valida = ~(s_cond.isna() | s_cond.isin(no_resp) | s_cond.isin(no_aplica))
            mask_inesperado = pd.Series(False, index=s_cond.index, dtype="boolean")
        else:
            mask_valida = s_cond.isin(codigos_validos)
            mask_inesperado = (
                s_cond.notna()
                & ~s_cond.isin(codigos_validos)
                & ~s_cond.isin(no_resp)
                & ~s_cond.isin(no_aplica)
            )

        n_valida = int(mask_valida.sum())
        n_inesperado = int(mask_inesperado.sum())
        pct = round(100 * n_valida / n_condicional, 2) if n_condicional > 0 else np.nan

        # CORRECCIÓN 4: Aserciones de cierre por trimestre
        assert n_valida + n_no_resp + n_no_aplica + n_faltante + n_inesperado == n_condicional, (
            f"No cierra {col} en {a}T{t}"
        )
        assert n_condicional + n_salto == n_general, (
            f"No cierra el universo de {col} en {a}T{t}"
        )

        filas.append({
            "ANO4": a,
            "TRIMESTRE": t,
            "n_universo_general": n_general,
            "n_universo_condicional": n_condicional,
            "n_salto_interno": n_salto,
            "n_respuesta_valida": n_valida,
            "n_no_respuesta": n_no_resp,
            "n_no_aplica_interno": n_no_aplica,
            "n_faltante_real": n_faltante,
            "n_codigo_inesperado": n_inesperado,
            "pct_cobertura": pct,
            "estado_filtro": info["estado_filtro"],
        })

    tabla = pd.DataFrame(filas)
    if not tabla.empty:
        tabla["variable"] = col
        tabla["universo_aplicable"] = info["universo"]
        tabla["tiene_filtro_condicional"] = info["filtro_condicional"] is not None
        tabla["cobertura_definitiva"] = info["estado_filtro"] in {"no requiere", "confirmada"}
    return tabla

# Variables a evaluar temporalmente (las que tienen entrada en UNIVERSOS)
vars_tiempo = list(UNIVERSOS.keys())
cobertura_periodo = pd.concat(
    [cobertura_por_periodo(v) for v in vars_tiempo if v in UNIVERSOS],
    ignore_index=True
)
display(cobertura_periodo)
cobertura_periodo.to_csv(OUT_DIR / "cobertura_variables_por_periodo.csv", index=False, encoding="utf-8-sig")



,ANO4,TRIMESTRE,n_universo_general,n_universo_condicional,n_salto_interno,n_respuesta_valida,n_no_respuesta,n_no_aplica_interno,n_faltante_real,n_codigo_inesperado,pct_cobertura,estado_filtro,variable,universo_aplicable,tiene_filtro_condicional,cobertura_definitiva
0,2023,4,9114,9114,0,7833,1281,0,0,0,85.94,no requiere,P21,todos,False,True
1,2024,1,8652,8652,0,7366,1286,0,0,0,85.14,no requiere,P21,todos,False,True
2,2024,2,8951,8951,0,7576,1375,0,0,0,84.64,no requiere,P21,todos,False,True
3,2024,3,9167,9167,0,7537,1630,0,0,0,82.22,no requiere,P21,todos,False,True
4,2024,4,9109,9109,0,7335,1774,0,0,0,80.52,no requiere,P21,todos,False,True
5,2025,1,8656,8656,0,7046,1610,0,0,0,81.40,no requiere,P21,todos,False,True
6,2025,2,8989,8989,0,7383,1606,0,0,0,82.13,no requiere,P21,todos,False,True
7,2025,3,8916,8916,0,7185,1731,0,0,0,80.59,no requiere,P21,todos,False,True
8,2025,4,8746,8746,0,7050,1696,0,0,0,80.61,no requiere,P21,todos,False,True
9,2023,4,9114,9114,0,9102,12,0,0,0,99.87,no requiere,PP3E_TOT,todos,False,True


## 8. Auditoría específica del ingreso

El ingreso de la ocupación principal (`P21`) es la variable central del futuro IVLR. Esta sección audita su comportamiento sin imputar valores y sin tratar el faltante como ingreso bajo. Se analizan: valores faltantes, códigos especiales, ceros, distribución y percentiles por periodo, y la estructura de faltantes por `CAT_OCUP`, `INTENSI`, `NIVEL_ED`, `TAM_EST` y periodo.

**Advertencia metodológica:** un ingreso faltante (no respuesta) no debe interpretarse como ingreso nulo ni como baja vulnerabilidad. La decisión sobre cómo tratar el faltante corresponde a la etapa de construcción del IVLR, fuera del alcance de esta auditoría.


In [91]:
# 8.1 Ingreso real heredado del pipeline de clustering
# `log_P21_real` ya fue construido en etapas previas (deflactación).
base_aud["P21_real"] = np.expm1(base_aud["log_P21_real"])
print("log_P21_real (heredado): n_obs =", base_aud["log_P21_real"].notna().sum())
print("P21_real (derivado):     n_obs =", base_aud["P21_real"].notna().sum())

# 8.2 Ceros, códigos especiales y faltante real
p21_raw = normalizar(base_aud["P21"])
print("\n=== Auditoría de P21 (ingreso de la ocupación principal) ===")
print("n total:", len(p21_raw))
print("n faltante (NaN):", int(p21_raw.isna().sum()))
print("n código especial (-9):", int((p21_raw == "-9").sum()))
print("n cero literal ('0'):", int((p21_raw == "0").sum()))
print("n positivo observado:", int(pd.to_numeric(p21_raw, errors="coerce").gt(0).sum()))
print("n faltante en log_P21_real:", int(base_aud["log_P21_real"].isna().sum()),
      f"({100 * base_aud['log_P21_real'].isna().mean():.2f}%)")

# 8.3 Origen exacto de los faltantes de log_P21_real
auditoria_faltantes_p21 = (
    base_aud.loc[
        base_aud["log_P21_real"].isna(),
        "P21"
    ]
    .value_counts(dropna=False)
    .rename_axis("valor_original_P21")
    .reset_index(name="n")
)
display(auditoria_faltantes_p21)
auditoria_faltantes_p21.to_csv(OUT_DIR / "auditoria_faltantes_P21.csv", index=False, encoding="utf-8-sig")
assert auditoria_faltantes_p21["n"].sum() == base_aud["log_P21_real"].isna().sum()

# 8.4 Cruce entre P21 original y faltante en log_P21_real
base_aud["log_P21_real_faltante"] = base_aud["log_P21_real"].isna()
cruce_p21_faltante = pd.crosstab(
    normalizar(base_aud["P21"]),
    base_aud["log_P21_real_faltante"],
    dropna=False,
).reset_index()
display(cruce_p21_faltante)
cruce_p21_faltante.to_csv(OUT_DIR / "cruce_p21_faltante.csv", index=False, encoding="utf-8-sig")

# 8.5 Percentiles por periodo (incluyendo ceros reales)
p21r = base_aud["P21_real"]
per_ingreso = (
    base_aud.assign(p21r=p21r)
    .dropna(subset=["p21r"])
    .groupby(["ANO4", "TRIMESTRE"])["p21r"]
    .agg(
        n="size",
        p25=lambda x: x.quantile(0.25),
        p50=lambda x: x.quantile(0.50),
        p75=lambda x: x.quantile(0.75),
        media="mean"
    )
    .round(0)
)
# Percentiles por periodo de P21_real observado.
# Se incluyen los ingresos nulos reales y se excluyen únicamente los NA.
display(per_ingreso)
per_ingreso.reset_index().to_csv(
    OUT_DIR / "percentiles_ingreso_por_periodo.csv",
    index=False,
    encoding="utf-8-sig",
)

# 8.6 Faltantes de ingreso real por grupo
log_real = base_aud["log_P21_real"]

def faltantes_por(col):
    return (
        base_aud.groupby(col, dropna=False)
        .apply(lambda g: pd.Series({
            "n": len(g),
            "pct_faltante_ingreso": 100 * log_real.loc[g.index].isna().mean()
        }), include_groups=False)
        .reset_index()
        .rename(columns={col: "grupo"})
        .assign(variable=col)
    )

faltantes_ingreso = pd.concat([
    faltantes_por("CAT_OCUP"),
    faltantes_por("INTENSI"),
    faltantes_por("NIVEL_ED"),
    faltantes_por("TAM_EST"),
    faltantes_por("cluster_final"),
    faltantes_por("ANO4"),
], ignore_index=True)
display(faltantes_ingreso)
faltantes_ingreso.to_csv(OUT_DIR / "faltantes_ingreso_por_grupo.csv", index=False, encoding="utf-8-sig")



log_P21_real (heredado): n_obs = 66311
P21_real (derivado):     n_obs = 66311

=== Auditoría de P21 (ingreso de la ocupación principal) ===
n total: 80300
n faltante (NaN): 0
n código especial (-9): 13989
n cero literal ('0'): 2942
n positivo observado: 63369
n faltante en log_P21_real: 13989 (17.42%)


,valor_original_P21,n
0,-9,13989


log_P21_real_faltante,P21,False,True
0,-9,0,13989
1,0,2942,0
2,1000,1,0
3,10000,226,0
4,100000,2933,0
...,...,...,...
768,982000,1,0
769,990000,2,0
770,99200,1,0
771,999999,1,0


n       p25       p50       p75     media
ANO4 TRIMESTRE                                              
2023 4          7833  167127.0  300829.0  501382.0  389604.0
2024 1          7366  143285.0  245631.0  409384.0  322724.0
     2          7576  129567.0  275329.0  453484.0  338397.0
     3          7537  143434.0  286867.0  502018.0  385341.0
     4          7335  164920.0  328527.0  525642.0  421089.0
2025 1          7046  219279.0  365464.0  609107.0  470236.0
     2          7383  226082.0  406947.0  678245.0  491314.0
     3          7185  214241.0  428481.0  642722.0  495666.0
     4          7050  200000.0  400000.0  700000.0  510813.0

,grupo,n,pct_faltante_ingreso,variable
0,1,1772.0,18.792325,CAT_OCUP
1,2,28245.0,20.683307,CAT_OCUP
2,3,49514.0,15.781395,CAT_OCUP
3,4,769.0,0.000000,CAT_OCUP
4,Subocupado,15937.0,16.935433,INTENSI
5,Ocupado pleno,42590.0,17.166001,INTENSI
6,Sobreocupado,20459.0,18.348893,INTENSI
7,No trabajó,1314.0,17.123288,INTENSI
8,1,3650.0,15.972603,NIVEL_ED
9,2,12820.0,18.307332,NIVEL_ED


## 9. Auditoría de intensidad e insatisfacción laboral

Se analizan las variables relacionadas con la intensidad horaria y el deseo/búsqueda de más horas. No se utiliza el término "subocupado" para valores de horas < 35, sino que se describe como "jornada inferior a 35 horas". Se presentan tablas cruzadas y medidas de asociación (V de Cramér).


In [67]:
INT = normalizar(base_aud["INTENSI"])
PP03G = normalizar(base_aud["PP03G"]).replace("0", np.nan)
PP03H = normalizar(base_aud["PP03H"]).replace("0", np.nan)
PP03I = normalizar(base_aud["PP03I"]).replace("0", np.nan)

# PP3E_TOT tratando 999 como faltante
PP3E = pd.to_numeric(
    normalizar(base_aud["PP3E_TOT"]),
    errors="coerce",
).replace(999, np.nan)

# Jornada inferior a 35 horas (con NA para valores faltantes)
jornada_menor_35 = pd.Series(
    pd.NA,
    index=PP3E.index,
    dtype="boolean",
)
mask_horas_validas = PP3E.notna()
jornada_menor_35.loc[mask_horas_validas] = PP3E.loc[mask_horas_validas] < 35

print("\n=== Tablas cruzadas de intensidad ===")
print("\nINTENSI vs PP03G (deseo de más horas):")
print(pd.crosstab(INT, PP03G, dropna=False).to_string())

print("\nINTENSI vs PP03I (búsqueda de más horas):")
print(pd.crosstab(INT, PP03I, dropna=False).to_string())

print("\nINTENSI vs PP03H (disponibilidad) [PROVISIONAL - filtro pendiente]:")
print(pd.crosstab(INT, PP03H, dropna=False).to_string())

# V de Cramér
def cramer_v(x, y):
    x = x.dropna().astype(str)
    y = y.dropna().astype(str)
    idx = x.index.intersection(y.index)
    x, y = x.loc[idx], y.loc[idx]
    if len(x) == 0 or x.nunique() < 2 or y.nunique() < 2:
        return np.nan
    tabla = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(tabla)
    n = tabla.to_numpy().sum()
    r, k = tabla.shape
    return float(np.sqrt((chi2 / n) / min(k - 1, r - 1)))

pares = [
    ("INTENSI vs PP03G", INT, PP03G),
    ("INTENSI vs PP03I", INT, PP03I),
    ("PP03G vs PP03I", PP03G, PP03I),
    ("PP03G vs PP03H", PP03G, PP03H),
    ("PP03H vs PP03I", PP03H, PP03I),
    ("INTENSI vs Jornada<35h", INT, jornada_menor_35),
]
filas_v = []
for nombre, x, y in pares:
    filas_v.append({"par": nombre, "cramer_v": round(cramer_v(x, y), 3)})
solapamiento = pd.DataFrame(filas_v)
print("\nV de Cramér entre variables de intensidad:")
display(solapamiento)
solapamiento.to_csv(OUT_DIR / "solapamiento_variables_laborales.csv", index=False, encoding="utf-8-sig")

print("\nHoras trabajadas (PP3E_TOT) por INTENSI:")
display(
    base_aud.assign(INTENSI=INT, PP3E_TOT=PP3E)
    .groupby("INTENSI")["PP3E_TOT"]
    .describe()
    .round(1)
)




=== Tablas cruzadas de intensidad ===

INTENSI vs PP03G (deseo de más horas):
PP03G              1      2   9  <NA>
INTENSI                              
No trabajó       332    980   2     0
Ocupado pleno   3445  39127  17     1
Sobreocupado    2118  18328  13     0
Subocupado     15922     14   1     0

INTENSI vs PP03I (búsqueda de más horas):
PP03I              1      2    9  <NA>
INTENSI                               
No trabajó       286   1024    4     0
Ocupado pleno   4174  38305  110     1
Sobreocupado    2176  18239   44     0
Subocupado     13834   2055   48     0

INTENSI vs PP03H (disponibilidad) [PROVISIONAL - filtro pendiente]:
PP03H              1    2    3   9   <NA>
INTENSI                                  
No trabajó       275   22   27   8    982
Ocupado pleno   2945   77  367  56  39145
Sobreocupado    1956   71   52  39  18341
Subocupado     15688  224    0  10     15

V de Cramér entre variables de intensidad:


,par,cramer_v
0,INTENSI vs PP03G,0.577
1,INTENSI vs PP03I,0.496
2,PP03G vs PP03I,0.652
3,PP03G vs PP03H,NaN
4,PP03H vs PP03I,0.202
5,INTENSI vs Jornada<35h,0.670



Horas trabajadas (PP3E_TOT) por INTENSI:


,count,mean,std,min,25%,50%,75%,max
INTENSI,,,,,,,,
No trabajó,1309.0,0.0,0.8,0.0,0.0,0.0,0.0,24.0
Ocupado pleno,42493.0,28.6,11.5,0.0,20.0,30.0,40.0,48.0
Sobreocupado,20409.0,54.2,12.5,5.0,48.0,50.0,60.0,168.0
Subocupado,15913.0,16.1,8.3,0.0,9.0,16.0,24.0,48.0


## 10. Auditoría de estabilidad

Se analizan variables de antigüedad y temporalidad, utilizando la auditoría de universos ya definida. No se armoniza aún; solo se documenta la cobertura y distribución.


In [68]:
def resumen_estabilidad(col, etiqueta):
    info = UNIVERSOS.get(col)
    
    if info is None:
        return {
            "variable": col,
            "etiqueta": etiqueta,
            "universo": "pendiente de definir",
            "n_universo_general": np.nan,
            "n_universo_condicional": np.nan,
            "n_respuesta_valida": np.nan,
            "n_no_respuesta": np.nan,
            "n_no_aplica_interno": np.nan,
            "n_faltante_real": np.nan,
            "n_codigo_inesperado": np.nan,
            "pct_cobertura": np.nan,
            "codificacion": "pendiente",
            "estado_filtro": "pendiente",
        }
    
    resultado = auditar_variable(col)
    
    return {
        "variable": col,
        "etiqueta": etiqueta,
        "universo": info["universo"],
        "n_universo_general": resultado["n_universo_general"],
        "n_universo_condicional": resultado["n_universo_condicional"],
        "n_respuesta_valida": resultado["n_respuesta_valida"],
        "n_no_respuesta": resultado["n_no_respuesta"],
        "n_no_aplica_interno": resultado["n_no_aplica_interno"],
        "n_faltante_real": resultado["n_faltante_real"],
        "n_codigo_inesperado": resultado["n_codigo_inesperado"],
        "pct_cobertura": resultado["pct_cobertura_universo"],
        "codificacion": info["codificacion"],
        "estado_filtro": info["estado_filtro"],
    }

variables_estabilidad = [
    ("PP05H", "Antigüedad continua de independientes"),
    ("PP07A", "Antigüedad continua de asalariados"),
    ("PP07C", "Existencia de tiempo de finalización"),
    ("PP07D", "Duración del trabajo temporario"),
]

estabilidad = pd.DataFrame([
    resumen_estabilidad(variable, etiqueta)
    for variable, etiqueta in variables_estabilidad
])

display(estabilidad)

estabilidad.to_csv(
    OUT_DIR / "auditoria_estabilidad_por_subuniverso.csv",
    index=False,
    encoding="utf-8-sig",
)

# Distribuciones por universo correspondiente
variables_distribucion = [
    (
        "PP05H",
        "PP05H, independientes",
        MASCARA_UNIVERSO["independientes"],
    ),
    (
        "PP07A",
        "PP07A, asalariados",
        MASCARA_UNIVERSO["asalariados"],
    ),
    (
        "PP07C",
        "PP07C, asalariados",
        MASCARA_UNIVERSO["asalariados"],
    ),
    (
        "PP07D",
        "PP07D, asalariados con filtro provisional",
        MASCARA_UNIVERSO["asalariados"],
    ),
]

for col, etiqueta, mascara in variables_distribucion:
    serie = normalizar(base_aud.loc[mascara, col])
    print(f"\n{etiqueta}:")
    print(serie.value_counts(dropna=False).sort_index().to_string())



,variable,etiqueta,universo,n_universo_general,n_universo_condicional,n_respuesta_valida,n_no_respuesta,n_no_aplica_interno,n_faltante_real,n_codigo_inesperado,pct_cobertura,codificacion,estado_filtro
0,PP05H,Antigüedad continua de independientes,independientes,30017,30017,29888,129,0,0,0,99.57,pendiente de confirmación documental,no requiere
1,PP07A,Antigüedad continua de asalariados,asalariados,49514,49514,38976,254,10284,0,0,78.72,confirmada,confirmado documentalmente; implementación pen...
2,PP07C,Existencia de tiempo de finalización,asalariados,49514,49514,30989,8241,10284,0,0,62.59,confirmada,confirmado documentalmente; implementación pen...
3,PP07D,Duración del trabajo temporario,asalariados,49514,18889,5326,13563,0,0,0,28.20,confirmada,confirmado



PP05H, independientes:
PP05H
1      666
2     1458
3     1167
4     2280
5    10310
6    14007
9      129

PP07A, asalariados:
PP07A
0    10284
1     2020
2     3820
3     3177
4     3603
5    16897
6     9459
9      254

PP07C, asalariados:
PP07C
0    10284
1    10648
2    20341
9     8241

PP07D, asalariados con filtro provisional:
PP07D
0    30625
1     1853
2      651
3      589
4      740
5     1493
9    13563


## 11. Variables con posible circularidad

Se identifican variables relacionadas con la definición de informalidad que podrían generar circularidad conceptual si se incorporaran al IVLR. Estas variables se documentan, pero no se proponen todavía como componentes del índice.


In [69]:
circulares = [
    ("PP07H", "descuento jubilatorio", "clasificación de asalariados informales", "posible circularidad; revisar metodología oficial"),
    ("PP07I", "aporte jubilatorio propio del asalariado", "variable vinculada con protección dentro del empleo asalariado", "posible circularidad; revisar metodología oficial"),
    ("PP05I", "registro de aportes (indep.)", "clasificación de independientes informales", "posible circularidad; revisar metodología oficial"),
    ("PP05J", "razón de no aportes", "complemento de PP05I", "pendiente de verificar contra la construcción de EMPLEO"),
    ("PP06A", "socios o familiares", "contexto de independientes", "pendiente de verificar contra la construcción de EMPLEO"),
    ("PP06E", "forma legal", "contexto de independientes", "pendiente de verificar contra la construcción de EMPLEO"),
]
tabla_circular = pd.DataFrame(circulares, columns=["variable", "descripcion", "uso_en_clasificacion", "riesgo_circularidad"])
display(tabla_circular)
tabla_circular.to_csv(OUT_DIR / "variables_con_posible_circularidad.csv", index=False, encoding="utf-8-sig")



,variable,descripcion,uso_en_clasificacion,riesgo_circularidad
0,PP07H,descuento jubilatorio,clasificación de asalariados informales,posible circularidad; revisar metodología oficial
1,PP07I,aporte jubilatorio propio del asalariado,variable vinculada con protección dentro del e...,posible circularidad; revisar metodología oficial
2,PP05I,registro de aportes (indep.),clasificación de independientes informales,posible circularidad; revisar metodología oficial
3,PP05J,razón de no aportes,complemento de PP05I,pendiente de verificar contra la construcción ...
4,PP06A,socios o familiares,contexto de independientes,pendiente de verificar contra la construcción ...
5,PP06E,forma legal,contexto de independientes,pendiente de verificar contra la construcción ...


## 12. Diccionario consolidado de candidatas

Se presenta una tabla que resume el estado de cada variable auditada, sin clasificarlas como "candidata principal" ni "excluir". Los estados posibles son:
- cobertura suficiente
- cobertura restringida a asalariados
- cobertura restringida a independientes
- codificación pendiente
- filtro condicional pendiente
- códigos inesperados pendientes de revisión
- variable contextual
- no disponible
- no comparable


In [82]:
# CORRECCIÓN 3: estado_neutral() priorizando el filtro
VARIABLES_CONTEXTO = {"P47T", "IPCF"}

def estado_neutral(row):
    # Prioridad 1: estado del filtro
    if row.get("estado_filtro") in {
        "pendiente de confirmación documental",
        "provisional",
    }:
        return "filtro condicional pendiente"

    # Prioridad 1b: filtro confirmado documentalmente con implementación pendiente
    if row.get("estado_filtro") == (
        "confirmado documentalmente; implementación pendiente"
    ):
        return "filtro confirmado documentalmente; implementación pendiente"

    # Prioridad 2: codificación pendiente
    if row.get("codificacion") == "pendiente de confirmación documental":
        return "codificación pendiente"

    # Prioridad 3: variable contextual
    if row["variable"] in VARIABLES_CONTEXTO:
        return "variable contextual"

    # Prioridad 4: códigos inesperados
    if row.get("n_codigo_inesperado", 0) > 0:
        return "códigos inesperados pendientes de revisión"

    # Prioridad 5: universo restringido
    if row["universo_aplicable"] == "independientes":
        return "cobertura restringida a independientes"

    if row["universo_aplicable"] == "asalariados":
        return "cobertura restringida a asalariados"

    # Prioridad 6: cobertura suficiente
    if row.get("pct_cobertura_universo", 0) >= 80:
        return "cobertura suficiente"

    return "no comparable"

# Construir diccionario a partir de la tabla de auditoría
diccionario = tabla_auditoria.copy()
diccionario["estado"] = diccionario.apply(estado_neutral, axis=1)

# Agregar descripción de variables (manual para las principales)
descripciones = {
    "P21": "Ingreso de la ocupación principal",
    "PP3E_TOT": "Horas trabajadas en la ocupación principal",
    "PP03G": "Deseo de trabajar más horas",
    "PP03H": "Disponibilidad para trabajar más horas",
    "PP03I": "Búsqueda de más horas en los últimos 30 días",
    "PP05H": "Antigüedad continua (independientes)",
    "PP07A": "Antigüedad continua (asalariados)",
    "PP07C": "Tiempo de finalización (temporalidad)",
    "PP07D": "Duración del trabajo temporario",
    "PP07H": "Descuento jubilatorio (asalariados)",
    "PP05I": "Registro de aportes (independientes)",
    "NIVEL_ED": "Nivel educativo",
}
diccionario["descripcion"] = diccionario["variable"].map(descripciones).fillna("pendiente de definir")

# Observación metodológica
def observacion_metodologica(row):
    if row["estado"] == "filtro condicional pendiente":
        return "La cobertura calculada es provisional hasta confirmar el recorrido del cuestionario"
    if row["estado"] == "codificación pendiente":
        return "Los códigos y su interpretación requieren verificación documental"
    if row.get("n_codigo_inesperado", 0) > 0:
        return f"Se detectaron {row['n_codigo_inesperado']} códigos no esperados; revisar"
    if row["estado"] == (
        "filtro confirmado documentalmente; implementación pendiente"
    ):
        return (
            "Excluir del universo condicional a los casos derivados desde "
            "servicio doméstico a 7f, una vez identificado de forma "
            "inequívoca el código correspondiente en la base."
        )
    return ""

diccionario["observacion_metodologica"] = diccionario.apply(observacion_metodologica, axis=1)

# Selección de columnas finales
diccionario = diccionario[[
    "variable", "descripcion", "universo_aplicable",
    "estado_filtro", "codificacion", "estado",
    "pct_cobertura_universo", "n_codigo_inesperado",
    "observacion_metodologica"
]]

display(diccionario)
diccionario.to_csv(OUT_DIR / "diccionario_candidatas_ivlr.csv", index=False, encoding="utf-8-sig")

,variable,descripcion,universo_aplicable,estado_filtro,codificacion,estado,pct_cobertura_universo,n_codigo_inesperado,observacion_metodologica
0,P21,Ingreso de la ocupación principal,todos,no requiere,confirmada,cobertura suficiente,82.58,0,
1,PP3E_TOT,Horas trabajadas en la ocupación principal,todos,no requiere,pendiente de confirmación documental,codificación pendiente,99.78,0,Los códigos y su interpretación requieren veri...
2,PP03G,Deseo de trabajar más horas,todos,no requiere,confirmada,cobertura suficiente,99.96,0,
3,PP03H,Disponibilidad para trabajar más horas,todos,confirmado,confirmada,cobertura suficiente,99.48,0,
4,PP03I,Búsqueda de más horas en los últimos 30 días,todos,no requiere,confirmada,cobertura suficiente,99.74,0,
5,PP05H,Antigüedad continua (independientes),independientes,no requiere,pendiente de confirmación documental,codificación pendiente,99.57,0,Los códigos y su interpretación requieren veri...
6,PP07A,Antigüedad continua (asalariados),asalariados,confirmado documentalmente; implementación pen...,confirmada,filtro confirmado documentalmente; implementac...,78.72,0,Excluir del universo condicional a los casos d...
7,PP07C,Tiempo de finalización (temporalidad),asalariados,confirmado documentalmente; implementación pen...,confirmada,filtro confirmado documentalmente; implementac...,62.59,0,Excluir del universo condicional a los casos d...
8,PP07D,Duración del trabajo temporario,asalariados,confirmado,confirmada,cobertura restringida a asalariados,28.20,0,
9,PP07H,Descuento jubilatorio (asalariados),asalariados,no requiere,confirmada,cobertura restringida a asalariados,100.00,0,


## 13. Tabla de filtros condicionales


In [81]:
# CORRECCIÓN 2: Tabla de filtros condicionales (según cuestionario oficial)
filtros = []

for var, info in UNIVERSOS.items():
    incluir = (
        info["filtro_condicional"] is not None
        or info["estado_filtro"]
        in {
            "provisional",
            "pendiente de confirmación documental",
            "confirmado documentalmente; implementación pendiente",
        }
    )
    
    if not incluir:
        continue
    
    if var == "PP03H":
        descripcion = "Filtro confirmado: PP03G == 1 (solo quienes desean más horas)"
    
    elif var == "PP07D":
        descripcion = "Filtro confirmado: PP07C en {1, 9}"
    
    elif var in {"PP07A", "PP07C"}:
        descripcion = (
            "Servicio doméstico salta desde 4b1 a 7f; "
            "excluir esos casos del universo condicional "
            "una vez identificado el código en la base"
        )
    
    else:
        descripcion = (
            "Filtro condicional pendiente "
            "de documentación"
        )
    
    res = next(
        (
            fila
            for fila in resultados_auditoria
            if fila["variable"] == var
        ),
        None,
    )
    
    if res is None:
        continue
    
    filtros.append({
        "variable": var,
        "universo_general": res["universo_aplicable"],
        "tiene_filtro_condicional": (
            info["filtro_condicional"] is not None
        ),
        "descripcion_filtro": descripcion,
        "estado_filtro": info["estado_filtro"],
        "n_universo_general": res["n_universo_general"],
        "n_universo_condicional": res["n_universo_condicional"],
        "n_salto_interno": res["n_salto_interno"],
        "observacion_metodologica": (
            info.get(
                "observacion",
                "La cobertura no se considera definitiva "
                "hasta confirmar documentalmente el recorrido "
                "del cuestionario.",
            )
        ),
    })

tabla_filtros = pd.DataFrame(filtros)

display(tabla_filtros)

tabla_filtros.to_csv(
    OUT_DIR / "auditoria_filtros_condicionales.csv",
    index=False,
    encoding="utf-8-sig",
)

,variable,universo_general,tiene_filtro_condicional,descripcion_filtro,estado_filtro,n_universo_general,n_universo_condicional,n_salto_interno,observacion_metodologica
0,PP03H,todos,True,Filtro confirmado: PP03G == 1 (solo quienes de...,confirmado,80300,21817,58483,La cobertura no se considera definitiva hasta ...
1,PP07A,asalariados,False,Servicio doméstico salta desde 4b1 a 7f; exclu...,confirmado documentalmente; implementación pen...,49514,49514,0,Excluir del universo condicional a los casos d...
2,PP07C,asalariados,False,Servicio doméstico salta desde 4b1 a 7f; exclu...,confirmado documentalmente; implementación pen...,49514,49514,0,Excluir del universo condicional a los casos d...
3,PP07D,asalariados,True,"Filtro confirmado: PP07C en {1, 9}",confirmado,49514,18889,30625,La cobertura no se considera definitiva hasta ...


## 14. Conclusiones y exportaciones

La auditoría no determina todavía la composición del IVLR. Las variables permanecen sujetas a evaluación de cobertura, comparabilidad, redundancia, circularidad y justificación conceptual. No se asignaron pesos, umbrales ni categorías de vulnerabilidad.


In [93]:
# CORRECCIÓN 13 y 14: Exportaciones y conclusión
versiones_entorno.to_csv(OUT_DIR / "versiones_entorno_vulnerabilidad.csv", index=False, encoding="utf-8-sig")

with open(OUT_DIR / "conclusiones_auditoria_vulnerabilidad.txt", "w", encoding="utf-8") as f:
    f.write(
        "Auditoría de variables candidatas para el IVLR en el empleo informal (2023T4-2025T4).\n"
        "El universo incluye 80.300 observaciones persona-trimestre de ocupados informales.\n"
        "Este notebook no construye el IVLR, no asigna pesos ni define categorías de vulnerabilidad.\n"
        "cluster_final se usa solo como contexto descriptivo.\n"
        "La auditoría identificó variables con distintos niveles de disponibilidad y cobertura.\n"
        "Su eventual incorporación al IVLR permanece pendiente de la confirmación documental de códigos y filtros,\n"
        "del tratamiento de los valores faltantes, de la comparación entre universos ocupacionales\n"
        "y del análisis de redundancia y circularidad.\n"
        "No se asignaron pesos, umbrales ni categorías de vulnerabilidad.\n"
    )

print("Archivos exportados en:", OUT_DIR)
for a in sorted(os.listdir(OUT_DIR)):
    print(" -", a)


Archivos exportados en: C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\08_vulnerabilidad
 - auditoria_codigos_y_universos.csv
 - auditoria_estabilidad_por_subuniverso.csv
 - auditoria_faltantes_P21.csv
 - auditoria_filtros_condicionales.csv
 - auditoria_variables_vulnerabilidad.csv
 - cobertura_variables_por_periodo.csv
 - conclusiones_auditoria_vulnerabilidad.txt
 - cruce_p21_faltante.csv
 - diccionario_candidatas_ivlr.csv
 - faltantes_ingreso_por_grupo.csv
 - percentiles_ingreso_por_periodo.csv
 - solapamiento_variables_laborales.csv
 - variables_con_posible_circularidad.csv
 - versiones_entorno_vulnerabilidad.csv


## Cierre documental de la auditoría

Para la ventana 2023T4–2025T4 se priorizan la Metodología INDEC N.º 43 y el diseño de registros vigente de la EPH. El cuestionario individual de 2019 se conserva únicamente como antecedente del flujo general, ya que es anterior a los cambios aplicados desde el cuarto trimestre de 2023 para medir informalidad laboral.

En particular, el personal de casas particulares recibe un tratamiento específico en la metodología y en el diseño de registros vigente. Por ello, no se utilizarán los saltos del cuestionario de 2019 para reconstruir automáticamente los universos de PP07A, PP07C o PP07D. Sus coberturas permanecen documentadas como brutas o provisionales hasta reconstruir el bloque correspondiente con las fuentes vigentes.

Esta decisión no afecta el clustering ni implica seleccionar variables para el IVLR.

In [2]:
# Registro documental de decisiones pendientes.
# Celda autosuficiente: no recalcula análisis ni modifica bases previas.

import os
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT_CIERRE = Path(
    os.getenv(
        "TFM_PROJECT_ROOT",
        r"C:\Users\tomas\OneDrive\Master Granada\TFM",
    )
).expanduser().resolve()

OUT_DIR_CIERRE = (
    PROJECT_ROOT_CIERRE
    / "Salidas"
    / "08_vulnerabilidad"
)

assert PROJECT_ROOT_CIERRE.exists(), (
    f"PROJECT_ROOT no encontrado: {PROJECT_ROOT_CIERRE}"
)

OUT_DIR_CIERRE.mkdir(
    parents=True,
    exist_ok=True,
)


cierre_documental = pd.DataFrame([
    {
        "tema": "fuente_prioritaria",
        "decision": (
            "Priorizar la Metodología INDEC N.º 43 y el diseño "
            "de registros vigente para 2023T4-2025T4."
        ),
        "estado": "confirmado",
    },
    {
        "tema": "cuestionario_2019",
        "decision": (
            "Conservar únicamente como antecedente. No utilizarlo "
            "para reconstruir automáticamente los filtros posteriores "
            "a los cambios aplicados desde 2023T4."
        ),
        "estado": "confirmado",
    },
    {
        "tema": "personal_casas_particulares",
        "decision": (
            "Tratar como bloque específico según la documentación "
            "vigente. No inferir su recorrido a partir del "
            "cuestionario de 2019."
        ),
        "estado": "pendiente de reconstrucción específica",
    },
    {
        "tema": "PP07A_PP07C_PP07D",
        "decision": (
            "Mantener sus coberturas como brutas o provisionales. "
            "No utilizar todavía estas variables para construir "
            "la dimensión de estabilidad del IVLR."
        ),
        "estado": "pendiente",
    },
])


display(cierre_documental)


RUTA_CIERRE = (
    OUT_DIR_CIERRE
    / "cierre_documental_auditoria_vulnerabilidad.csv"
)

cierre_documental.to_csv(
    RUTA_CIERRE,
    index=False,
    encoding="utf-8-sig",
)

assert RUTA_CIERRE.exists(), (
    f"No se generó el archivo esperado: {RUTA_CIERRE}"
)

assert len(cierre_documental) == 4

print(f"Archivo exportado: {RUTA_CIERRE}")

,tema,decision,estado
0,fuente_prioritaria,Priorizar la Metodología INDEC N.º 43 y el dis...,confirmado
1,cuestionario_2019,Conservar únicamente como antecedente. No util...,confirmado
2,personal_casas_particulares,Tratar como bloque específico según la documen...,pendiente de reconstrucción específica
3,PP07A_PP07C_PP07D,Mantener sus coberturas como brutas o provisio...,pendiente


Archivo exportado: C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\08_vulnerabilidad\cierre_documental_auditoria_vulnerabilidad.csv
